# 02 · Acondicionamiento de la señal

**Proyecto final · Reactividad alfa en EEG durante la alternancia de ojos abiertos y cerrados**

| | |
|---|---|
| Dueño | B · Adrián |
| Etapas de la rúbrica | Etapa 2 (15 %) |
| Entregable técnico | `acondicionar(raw, cfg)` |
| Figuras | fig02_original_vs_acondicionada (mismo fragmento en el tiempo + PSD antes y después) |

Reglas: este notebook solo lo edita su dueño. Se abre desde GitHub, se guarda con *Archivo → Guardar una copia en GitHub* en tu rama
(nunca en `main`) y antes de guardar se borran las salidas (*Editar → Borrar todas las salidas*). Las figuras se guardan con
`alfa.guardar_figura(fig, "figNN_nombre")` en la carpeta `figuras/` de Drive.

## 0. Preparación

In [ ]:
# ── Preparación (igual en todos los notebooks; no editar) ──────────────
RAMA = "main"   # cámbiala solo si necesitas un alfa.py que aún está en un PR

import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/repo"
    URL = "https://github.com/moiseiseiseis/Alpha-reactivity-during-resting-state-EEG.git"
    if not os.path.isdir(REPO):
        !git clone -q -b {RAMA} {URL} {REPO}
    else:
        !git -C {REPO} fetch -q origin {RAMA} && git -C {REPO} checkout -q {RAMA} && git -C {REPO} reset -q --hard origin/{RAMA}
    %pip install -q mne
    sys.path.insert(0, f"{REPO}/src")
else:
    sys.path.insert(0, os.path.abspath("../src"))

import importlib
import alfa
importlib.reload(alfa)   # recoge cambios de alfa.py sin reiniciar el entorno

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

R = alfa.rutas()
print("Datos en:", R["datos"])
print("Registro encontrado:", R["vhdr"].exists())

# Parámetros en uso: todo sale de CONFIG (no escribir números a mano en el notebook)
cfg = alfa.CONFIG
alfa.imprimir_config(cfg)

### Carga del registro

Usa las funciones del carril A. Mientras `cargar_raw()` / `tabla_bloques()` no estén en `main`, cae a una carga mínima con MNE (VEOG como `eog`) para que este notebook corra de forma independiente.

In [ ]:
try:
    raw = alfa.cargar_raw(R["vhdr"])
except NotImplementedError:
    print("AVISO: alfa.cargar_raw() aún no está en main; carga provisional con MNE (VEOG como 'eog').")
    raw = mne.io.read_raw_brainvision(R["vhdr"], eog=("VEOG",), preload=True, verbose=False)

try:
    bloques = alfa.tabla_bloques(raw)
except NotImplementedError:
    print("AVISO: alfa.tabla_bloques() pendiente; la Figura 2 no sombreará bloques EO/EC.")
    bloques = None

print(f"fs = {raw.info['sfreq']:g} Hz | canales = {len(raw.ch_names)} | duración = {raw.times[-1]:.1f} s")
print(f"Filtros de hardware del registro: pasa-altas {raw.info['highpass']} Hz, pasa-bajas {raw.info['lowpass']} Hz")

## 1. Qué necesita la señal

- [x] PSD de la señal original: deriva lenta, 50 Hz y armónicos, ruido de alta frecuencia
- [x] Qué parte del espectro importa para alfa (8–13 Hz) y qué no

In [ ]:
# ── Parámetros de ESTE notebook ────────────────────────────────────────
# Son criterios de visualización y verificación; NO cambian el pipeline (eso vive en CONFIG).
from scipy.integrate import trapezoid

FREQ_LINEA      = 50.0                      # Hz, red eléctrica (registro adquirido en Alemania)
BANDA_DERIVA    = (0.1, 1.0)                # Hz, deriva lenta / sudor / DC residual
BANDA_LINEA     = (FREQ_LINEA - 1.0, FREQ_LINEA + 1.0)
BANDA_ALTA      = (52.0, 70.0)              # Hz, ruido de alta frecuencia (EMG) fuera de la línea
BANDAS = {"deriva": BANDA_DERIVA, "alfa": tuple(cfg["banda_alfa"]),
          "linea": BANDA_LINEA, "alta": BANDA_ALTA}
FMAX_PSD        = 70.0                      # Hz, límite superior de las PSD mostradas
PSD_VENTANA_S   = 4.0                       # s por segmento de Welch -> resolución 0.25 Hz
FRAGMENTO_S     = 20.0                      # s del fragmento temporal de la Figura 2
CANAL_FIG       = "Oz"                      # canal de la Figura 2 (si no existe, el primero de CONFIG)
UMBRAL_LINEA_DB = -30.0                     # línea/alfa por debajo de esto => notch innecesario
TOL_ALFA_DB     = 0.5                       # |Δ potencia alfa| máxima aceptable por el acondicionamiento
R_MIN_FORMA     = 0.99                      # correlación mínima de la onda alfa antes/después

# ── Funciones auxiliares (solo de este notebook) ───────────────────────
def canales_analisis(raw, cfg):
    # Canales de CONFIG presentes en el registro (avisa de los que falten).
    presentes = [c for c in cfg["canales"] if c in raw.ch_names]
    faltan = [c for c in cfg["canales"] if c not in raw.ch_names]
    if faltan:
        print("AVISO: canales de CONFIG ausentes en el registro:", faltan)
    return presentes

def psd_welch(raw, canales):
    # PSD de Welch (Hamming, 50 % de solape) en µV²/Hz; devuelve (n_canales x n_freqs, freqs).
    n_fft = int(round(PSD_VENTANA_S * raw.info["sfreq"]))
    psd = raw.compute_psd(method="welch", fmin=BANDA_DERIVA[0], fmax=FMAX_PSD, picks=canales,
                          n_fft=n_fft, n_overlap=n_fft // 2, window="hamming",
                          average="mean", verbose=False)
    datos, freqs = psd.get_data(return_freqs=True)
    return datos * 1e12, freqs                      # V²/Hz -> µV²/Hz

def potencia_banda(psd, freqs, banda):
    # Potencia (µV²) por canal: integral trapezoidal de la PSD en la banda.
    m = (freqs >= banda[0]) & (freqs <= banda[1])
    return trapezoid(psd[:, m], freqs[m], axis=-1)

def potencias(psd, freqs, canales):
    return pd.DataFrame({n: potencia_banda(psd, freqs, b) for n, b in BANDAS.items()}, index=canales)

def a_db(x, ref):
    return 10.0 * np.log10(np.asarray(x) / np.asarray(ref))

In [ ]:
canales = canales_analisis(raw, cfg)
psd_pre, f_pre = psd_welch(raw, canales)
bp_pre = potencias(psd_pre, f_pre, canales)

diag = pd.DataFrame({
    "deriva (0.1–1 Hz) / alfa [dB]":  a_db(bp_pre["deriva"], bp_pre["alfa"]),
    "línea (49–51 Hz) / alfa [dB]":   a_db(bp_pre["linea"],  bp_pre["alfa"]),
    "alta frec. (52–70 Hz) / alfa [dB]": a_db(bp_pre["alta"], bp_pre["alfa"]),
}, index=canales)
diag.loc["MEDIA"] = diag.mean()
print("Señal ORIGINAL: potencia de cada componente relativa a la potencia alfa (8–13 Hz)")
print(diag.round(1).to_string())

**Lectura.** Valores positivos indican que el componente tiene más potencia que alfa; valores negativos, menos. La deriva (< 1 Hz) y la línea de 50 Hz son interferencias que no pertenecen al ritmo alfa y se pueden eliminar sin tocar 8–13 Hz, porque están separadas de la banda de interés por más de una década (deriva) o por 37 Hz (línea).

## 2. Cadena de acondicionamiento

- [x] Pasa-altas: frecuencia de corte y por qué
- [x] Pasa-bajas: frecuencia de corte y por qué
- [x] Notch 50 Hz: ¿hace falta si ya hay pasa-bajas a 40 Hz?
- [x] Remuestreo a `sfreq_analisis`: orden respecto al filtrado (antialiasing)
- [x] Tipo de filtro (FIR/IIR), orden, fase

In [ ]:
raw_ac = alfa.acondicionar(raw, cfg)

# Variante complementaria (con/sin notch) SOLO para decidir si el notch hace falta.
notch_cfg = cfg["filtro"]["notch"]
raw_alt = alfa.acondicionar(raw, alfa.variante(filtro={"notch": None if notch_cfg is not None else FREQ_LINEA}))
con_notch, sin_notch = (raw_ac, raw_alt) if notch_cfg is not None else (raw_alt, raw_ac)

psd_sin, f_ac = psd_welch(sin_notch, canales)
psd_con, _    = psd_welch(con_notch, canales)
bp_sin, bp_con = potencias(psd_sin, f_ac, canales), potencias(psd_con, f_ac, canales)

decision = pd.DataFrame({
    "línea/alfa SIN notch [dB]": a_db(bp_sin["linea"], bp_sin["alfa"]),
    "línea/alfa CON notch [dB]": a_db(bp_con["linea"], bp_con["alfa"]),
    "Δ alfa con vs sin notch [dB]": a_db(bp_con["alfa"], bp_sin["alfa"]),
}, index=canales)
print(f"Pasa-banda {cfg['filtro']['l_freq']}–{cfg['filtro']['h_freq']} Hz ya aplicado en ambas variantes.")
print(decision.round(2).to_string())

NOTCH_NECESARIO = bool(decision["línea/alfa SIN notch [dB]"].max() > UMBRAL_LINEA_DB)
print(f"\nPeor canal sin notch: {decision['línea/alfa SIN notch [dB]'].max():.1f} dB (umbral {UMBRAL_LINEA_DB:.0f} dB)")
print("DECISIÓN: notch de 50 Hz", "NECESARIO" if NOTCH_NECESARIO else "NO necesario (redundante con el pasa-bajas)")
if NOTCH_NECESARIO != (notch_cfg is not None):
    print("AVISO: CONFIG['filtro']['notch'] no coincide con la decisión; ajustar por PR a src/alfa.py.")

### Filtros elegidos

> **Pasa-banda FIR de fase cero, 1–40 Hz (Hamming, `firwin`), aplicado a la frecuencia original y seguido de remuestreo a 250 Hz.**
>
> - **Pasa-altas 1 Hz.** Elimina deriva de línea base, offset residual y artefactos de sudor/electrodo, que concentran su potencia por debajo de ~1 Hz y, al ser lentos, desestabilizarían la PSD en la banda baja. Está a más de una octava de 8 Hz, así que no afecta alfa. *Se pierde:* actividad delta (< 1 Hz), que no interviene en este análisis.
> - **Pasa-bajas 40 Hz.** Conserva alfa (8–13 Hz) y beta con amplio margen, atenúa el ruido muscular (EMG) y la línea de 50 Hz, y actúa de antialiasing antes de remuestrear. *Se pierde:* gamma y todo lo que quede por encima de 40 Hz.
> - **Notch 50 Hz: según la celda de decisión.** Con el pasa-bajas a 40 Hz, la atenuación del filtro en 50 Hz es de varias decenas de dB (valor exacto en §4). El notch solo se activa si la línea residual supera `UMBRAL_LINEA_DB` respecto a alfa. Además, un notch estrecho no aporta nada sobre un pasa-bajas que ya cae en esa frecuencia, y añade un segundo filtro a documentar.
> - **FIR de fase cero (no causal).** Sin distorsión de fase ni retardo de grupo, de modo que los instantes de transición EO/EC no se desplazan. El análisis es *offline*, por lo que la no causalidad no es un problema. *Costo:* núcleo de ≈ 3.3 s, con efectos de borde en los primeros y últimos ≈ 1.7 s del registro (MNE rellena con reflexión).
> - **Filtrar antes de remuestrear.** El pasa-bajas a 40 Hz queda muy por debajo del Nyquist de salida (125 Hz), así que no hay *aliasing* al pasar a 250 Hz y se evita filtrar con un núcleo mucho más largo a baja frecuencia relativa.

## 3. Figura 2 · Original vs acondicionada

- [x] Mismo fragmento en el tiempo
- [x] PSD antes y después, con la banda 8–13 Hz marcada

In [ ]:
ch = CANAL_FIG if CANAL_FIG in canales else canales[0]

# Fragmento centrado en la primera transición de condición (si hay tabla de bloques)
t_centro = 60.0
if bloques is not None and len(bloques) > 1:
    cambios = bloques.index[bloques["condicion"] != bloques["condicion"].shift()][1:]
    if len(cambios):
        t_centro = float(bloques.loc[cambios[0], "inicio_s"])
t0 = max(0.0, t_centro - FRAGMENTO_S / 2)
t1 = min(float(raw.times[-1]), t0 + FRAGMENTO_S)

def fragmento(r, canal, t0, t1):
    x, t = r.get_data(picks=[canal], tmin=t0, tmax=t1, return_times=True)
    return t, x[0] * 1e6                              # V -> µV

t_pre,  x_pre  = fragmento(raw,    ch, t0, t1)
t_post, x_post = fragmento(raw_ac, ch, t0, t1)
f_h, mag_db, h = alfa.respuesta_filtro(raw.info["sfreq"], cfg)

plt.rcParams.update({"font.size": 10, "axes.titlesize": 11})
fig = plt.figure(figsize=(11, 8.5), constrained_layout=True)
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.1])
ax_t, ax_p, ax_h = fig.add_subplot(gs[0, :]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])

# (a) Dominio del tiempo: mismo fragmento, original vs acondicionada
NOMBRE_COND = {"EO": "Ojos abiertos (EO)", "EC": "Ojos cerrados (EC)"}
COLOR_COND  = {"EO": "#2ca02c", "EC": "#ff7f0e"}
if bloques is not None:
    vistos = set()
    for _, b in bloques.iterrows():
        a, z = max(b["inicio_s"], t0), min(b["fin_s"], t1)
        if z > a:
            ax_t.axvspan(a, z, color=COLOR_COND.get(b["condicion"], "gray"), alpha=0.12, lw=0,
                         label=NOMBRE_COND.get(b["condicion"], b["condicion"]) if b["condicion"] not in vistos else None)
            vistos.add(b["condicion"])
ax_t.plot(t_pre,  x_pre,  color="#7f7f7f", lw=0.8, label=f"Original ({raw.info['sfreq']:g} Hz)")
ax_t.plot(t_post, x_post, color="#1f77b4", lw=1.0, label=f"Acondicionada ({raw_ac.info['sfreq']:g} Hz)")
ax_t.set_title(f"(a) Canal {ch}: mismo fragmento antes y después del acondicionamiento")
ax_t.set_xlabel("Tiempo (s)"); ax_t.set_ylabel("Amplitud (µV)")
ax_t.set_xlim(t0, t1); ax_t.grid(True, alpha=0.3); ax_t.legend(loc="upper right", ncol=2, fontsize=8)

# (b) PSD promedio de los canales de análisis, antes y después
ax_p.semilogy(f_pre, psd_pre.mean(axis=0), color="#7f7f7f", lw=1.3, label="Original")
ax_p.semilogy(f_ac,  (psd_con if notch_cfg is not None else psd_sin).mean(axis=0),
              color="#1f77b4", lw=1.3, label="Acondicionada")
ax_p.axvspan(*cfg["banda_alfa"], color="#d62728", alpha=0.15, label=f"Banda alfa {cfg['banda_alfa'][0]:g}–{cfg['banda_alfa'][1]:g} Hz")
ax_p.axvline(FREQ_LINEA, color="k", ls="--", lw=0.8, label=f"Línea eléctrica ({FREQ_LINEA:g} Hz)")
ax_p.set_title(f"(b) DEP promedio ({len(canales)} canales)")
ax_p.set_xlabel("Frecuencia (Hz)"); ax_p.set_ylabel("DEP (µV²/Hz)")
ax_p.set_xlim(0, FMAX_PSD); ax_p.grid(True, which="both", alpha=0.3); ax_p.legend(fontsize=8)

# (c) Respuesta en frecuencia del pasa-banda
ax_h.plot(f_h, mag_db, color="#1f77b4", lw=1.3, label="Pasa-banda FIR de fase cero")
ax_h.axvspan(*cfg["banda_alfa"], color="#d62728", alpha=0.15, label="Banda alfa")
ax_h.axhline(-3, color="gray", ls=":", lw=0.9, label="−3 dB")
ax_h.axvline(FREQ_LINEA, color="k", ls="--", lw=0.8, label=f"{FREQ_LINEA:g} Hz")
ax_h.set_title("(c) Respuesta en frecuencia del filtro")
ax_h.set_xlabel("Frecuencia (Hz)"); ax_h.set_ylabel("Magnitud (dB)")
ax_h.set_xlim(0, FMAX_PSD); ax_h.set_ylim(-110, 5); ax_h.grid(True, alpha=0.3); ax_h.legend(fontsize=8)

fig.suptitle("Figura 2. EEG original vs acondicionada (sub-010002)", fontsize=13, fontweight="bold")
alfa.guardar_figura(fig, "fig02_original_vs_acondicionada", dpi=300)
plt.show()

## 4. ¿El acondicionamiento altera el alfa?

- [x] Comparar potencia 8–13 Hz antes y después
- [x] Respuesta en frecuencia del filtro en la banda alfa
- [x] Qué información se pierde o se altera

In [ ]:
psd_post, f_post = psd_welch(raw_ac, canales)
bp_post = potencias(psd_post, f_post, canales)
tot = lambda p, f: potencia_banda(p, f, (cfg["filtro"]["l_freq"], cfg["filtro"]["h_freq"]))

# Forma de onda alfa: banda 8–13 Hz del original (remuestreado) vs de la acondicionada
fs_ana = cfg["sfreq_analisis"]
a_pre  = (raw.copy().pick(canales).filter(*cfg["banda_alfa"], method="fir", phase="zero", verbose=False)
             .resample(fs_ana, verbose=False).get_data())
a_post = raw_ac.copy().pick(canales).filter(*cfg["banda_alfa"], method="fir", phase="zero", verbose=False).get_data()
n = min(a_pre.shape[1], a_post.shape[1])
r_forma = np.array([np.corrcoef(a_pre[i, :n], a_post[i, :n])[0, 1] for i in range(len(canales))])

std_pre  = raw.get_data(picks=canales).std(axis=1) * 1e6
std_post = raw_ac.get_data(picks=canales).std(axis=1) * 1e6

met = pd.DataFrame({
    "std_pre_uV": std_pre, "std_post_uV": std_post,
    "alfa_pre_uV2": bp_pre["alfa"], "alfa_post_uV2": bp_post["alfa"],
    "delta_alfa_dB": a_db(bp_post["alfa"], bp_pre["alfa"]),
    "alfa_rel_pre": bp_pre["alfa"] / tot(psd_pre, f_pre), "alfa_rel_post": bp_post["alfa"] / tot(psd_post, f_post),
    "atenuacion_deriva_dB": a_db(bp_post["deriva"], bp_pre["deriva"]),
    "atenuacion_linea_dB": a_db(bp_post["linea"], bp_pre["linea"]),
    "alfa/linea_pre_dB": a_db(bp_pre["alfa"], bp_pre["linea"]),
    "alfa/linea_post_dB": a_db(bp_post["alfa"], bp_post["linea"]),
    "r_forma_alfa": r_forma,
}, index=canales)
met.index.name = "canal"
resumen_met = met.mean().to_frame("MEDIA").T
print("Métricas por canal (antes -> después del acondicionamiento)")
print(pd.concat([met, resumen_met]).round(3).to_string())

cumple = (met["delta_alfa_dB"].abs().max() < TOL_ALFA_DB) and (met["r_forma_alfa"].min() > R_MIN_FORMA)
print(f"\nCriterio de preservación de alfa: |Δ| < {TOL_ALFA_DB} dB y r > {R_MIN_FORMA} ->", "CUMPLE" if cumple else "NO CUMPLE")

# Respuesta del filtro (la misma de la Figura 2c)
fs0 = raw.info["sfreq"]
en_db = lambda f0: float(mag_db[np.argmin(np.abs(f_h - f0))])
ok3 = np.where(mag_db >= -3)[0]
m_alfa = (f_h >= cfg["banda_alfa"][0]) & (f_h <= cfg["banda_alfa"][1])
filtro_tab = pd.DataFrame([
    ("Tipo", "FIR fase cero, Hamming (firwin)"),
    ("Orden (taps)", len(h)),
    ("Duración del núcleo (s)", round(len(h) / fs0, 2)),
    ("Retardo de grupo (s)", 0.0),
    ("Frecuencia −3 dB inferior (Hz)", round(float(f_h[ok3[0]]), 2)),
    ("Frecuencia −3 dB superior (Hz)", round(float(f_h[ok3[-1]]), 2)),
    ("Rizado máx. en 8–13 Hz (dB)", round(float(np.abs(mag_db[m_alfa]).max()), 4)),
    ("Magnitud a 0.5 Hz (dB)", round(en_db(0.5), 1)),
    ("Magnitud a 50 Hz (dB)", round(en_db(FREQ_LINEA), 1)),
    ("fs original -> análisis (Hz)", f"{fs0:g} -> {raw_ac.info['sfreq']:g}"),
], columns=["parametro", "valor"])
print("\nCaracterísticas del filtro")
print(filtro_tab.to_string(index=False))

# Exportar a resultados/ (Drive; nunca al repo)
met.to_csv(R["resultados"] / "02_metricas_por_canal.csv")
filtro_tab.to_csv(R["resultados"] / "02_metricas_filtro.csv", index=False)
decision.to_csv(R["resultados"] / "02_decision_notch.csv")
print("\nCSV guardados en:", R["resultados"])

## Interpretación breve

In [ ]:
# Texto para el informe con los valores medidos arriba (copiar a la sección «Estrategia de acondicionamiento»)
import textwrap
txt = (
    f"El registro, adquirido a {fs0:g} Hz, se acondicionó con un filtro FIR de fase cero (Hamming) "
    f"pasa-banda de {cfg['filtro']['l_freq']:g}–{cfg['filtro']['h_freq']:g} Hz y se remuestreó a {raw_ac.info['sfreq']:g} Hz. "
    f"El filtro presenta una magnitud de {en_db(FREQ_LINEA):.1f} dB en {FREQ_LINEA:g} Hz y un rizado máximo de "
    f"{np.abs(mag_db[m_alfa]).max():.3f} dB en la banda alfa. "
    f"La potencia alfa cambió en promedio {met['delta_alfa_dB'].mean():+.2f} dB (máximo absoluto {met['delta_alfa_dB'].abs().max():.2f} dB) "
    f"y la forma de onda alfa se conservó con r ≥ {met['r_forma_alfa'].min():.4f}. "
    f"La deriva (0.1–1 Hz) se atenuó {met['atenuacion_deriva_dB'].mean():.1f} dB y la línea de {FREQ_LINEA:g} Hz "
    f"{met['atenuacion_linea_dB'].mean():.1f} dB en promedio. "
    f"Notch de {FREQ_LINEA:g} Hz: {'necesario' if NOTCH_NECESARIO else 'innecesario'} "
    f"(línea/alfa sin notch: {decision['línea/alfa SIN notch [dB]'].max():.1f} dB en el peor canal)."
)
print(textwrap.fill(txt, 110))

> _Observación (completar tras ejecutar):_ el acondicionamiento elimina deriva y línea sin alterar de forma apreciable la potencia ni la forma de onda alfa (ver tabla de §4); lo que se pierde es la información < 1 Hz y > 40 Hz, ajena a este análisis.